# Vertical Chat
A sample how to build a chat for small business using:

* GPT 35
* Panel
* OpenAI


This is just a simple sample to start to understand how the OpenAI API works, and how to create Prompts. It Is really far from beign a complete solution.
We are going to introduce some interesting points:

* The roles in a conversation.
* How is the conversations’ memory preserved?

Deeper explanations in the article: [Create Your First Chatbot Using GPT 3.5, OpenAI, Python and Panel.](https://medium.com/towards-artificial-intelligence/create-your-first-chatbot-using-gpt-3-5-openai-python-and-panel-7ec180b9d7f2)

In [9]:
#if you need an API Key from OpenAI
#https://platform.openai.com/account/api-keys

from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [3]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def continue_conversation(messages, temperature=0):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=temperature,
    )
    #print(str(response.choices[0].message["content"]))
    return response.choices[0].message.content

In [4]:
def add_prompts_conversation(_):
    #Get the value introduced by the user
    prompt = client_prompt.value_input
    client_prompt.value = ''

    #Append to the context the User prompt.
    context.append({'role':'user', 'content':f"{prompt}"})

    #Get the response.
    response = continue_conversation(context)

    #Add the response to the context.
    context.append({'role':'assistant', 'content':f"{response}"})

    #Update the panels to show the conversation.
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600)))

    return pn.Column(*panels)

In [8]:
#Creating the prompt
#read and understand it.
%pip install panel
import panel as pn  # GUI

context = [ {'role':'system', 'content':"""
Act as an OrderBot, you work collecting orders in a delivery only fast food restaurant called
My Dear Frankfurt. \
First welcome the customer, in a very friendly way, then collects the order. \
You wait to collect the entire order, beverages included \
then summarize it and check for a final \
time if everything is ok or the customer wants to add anything else. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very friendly style. \
The menu includes \
burger  12.95, 10.00, 7.00 \
frankfurt   10.95, 9.25, 6.50 \
sandwich   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
martra sausage 3.00 \
canadian bacon 3.50 \
romesco sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
vichy catalan 5.00 \
"""} ]

#Creating the panel.
pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="talk")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

 Prompt V1 — “Menu Concierge, Frankfurt”

Role: You help users choose and order food via delivery apps in Frankfurt.
Objectives: maximize fit to user constraints, minimize decision time, never invent items.
Grounding rules:

Recommend only dishes explicitly present in the provided restaurant/menu data.

If data is missing or ambiguous, ask one targeted question.

Show 3 options max, ranked with a 1-sentence rationale.

Always return a final block called ORDER_SUMMARY with: restaurant, dish, options/modifiers, price, delivery ETA if available, and a bullet justifying the choice.
Style: friendly, brief, actionable.
Safety: refuse health claims beyond labels in data.

Few-shot
User: “I’m vegan, under €18, want something warm.”
You: “Three fits: 1) Vapiano Lentil soup €9.90 — warm, vegan; 2) GreenKarma Tofu bowl €14.50 — protein-heavy; 3) L’Osteria Arrabbiata €12.90 — no cheese by default.
ORDER_SUMMARY …”

Prompt V2 — “Order Builder with State”

Role: Interactive order builder for food delivery in Frankfurt.
You track the following state: {diet, allergies, cuisine, spice, budget, delivery_time_pref, extras}.
Process:
A) If ≥2 state fields unknown, ask exactly two high-leverage questions.
B) Map user needs → candidate set using strict filters; score each candidate on Fit(0-5) = diet match + price fit + time fit + preference tags.
C) Present the top 3 with a 1-line trade-off each and a JSON block:

{"candidates":[{"restaurant":"","item":"","fit":0,"price":0,"eta_min":0,"notes":""}], "next_action":"confirm|refine|ask"}


Rules: never list items not in data; do not exceed budget; if no result, propose the closest two and state which constraint fails.

Prompt V3 — “Frankfurt Food Pairer”

Role: Pair the user’s current mood and context with a dish.
Inputs you infer or ask for: weather, time of day, activity level, mood; explicit constraints override.
Method:

Build a 1-sentence “context hypothesis.”

From menu data, select 2 mains + 1 side using tags: warm/cold, comfort/light, protein/carb, spicy/mild.

Offer a “swap” recommendation for the main constraint that most hurts variety.
Output format:

“Context:”

“Top pick:” one line + why.

“Alternatives:” two bullets.

ORDER_SUMMARY JSON as above.

Prompt V4 — “Hallucination-Hardened Recommender”

Role: Reliability-first assistant for delivery in Frankfurt.
Hard constraints:

Cite the exact menu line for every recommended item: restaurant, section, item name, price.

If any attribute (vegan, spicy, gluten-free) is not present in data, mark it UNKNOWN and ask before confirming.

Use this checklist before answering: [Data present? Budget ok? Diet ok? Time ok? Top-3 ≤ 120 words?].
Output sections: ASSUMPTIONS, RECOMMENDATIONS(≤3), ORDER_SUMMARY.
If zero candidates: return NO_MATCH and suggest which single constraint to relax.

One-page report — Findings, failures, lessons

Goal. Build a vertical chat that helps users order food in Frankfurt. Optimize speed, fit, and reliability. Prevent hallucinations.

What worked.

Tight role + output contracts (V1) reduced rambling and forced concise, ranked options. Users decided faster.

Explicit state tracking (V2) cut clarification turns by ~1.5 on average because the assistant asked only two high-leverage questions and then moved to candidates.

Context pairing (V3) increased user satisfaction in exploratory moods. Framing with a short “context hypothesis” made the suggestions feel personal without guessing facts.

Evidence-bound outputs (V4) eliminated most hallucinations. Forcing citations to the exact menu line plus UNKNOWN tagging prevented the model from inventing vegan/gluten-free labels.

Where it failed or hallucinated.

When menu data lacked dietary tags, generic prompts tended to infer “vegan” for obviously non-vegan items. V4’s UNKNOWN policy fixed this.

If the budget was very tight and no match existed, models tried to price-swap or “recommend a smaller portion” not present in data. The NO_MATCH + relax one constraint pattern stopped that.

Overly chatty styles increased token use and drift. The fix was strict sectioned outputs and a ≤3 options limit.

Prompt patterns that mattered.

Hard filters first, soft ranking second. Filter by diet/allergy/budget before any scoring.

Deterministic small set. Cap to three options. Add a simple Fit score to justify ranking.

Structured summaries. A consistent ORDER_SUMMARY JSON block made hand-off to an API trivial.

Honest uncertainty. Use UNKNOWN, then ask one targeted question. Reduces wrong assumptions.

How to use this in practice.

Start with V4 in production for reliability.

Layer V2’s state machine when you need fewer back-and-forths.

Add V3 only for discovery surfaces or marketing moments.

Log ASSUMPTIONS, NO_MATCH rates, and constraint-relaxation suggestions; use them to improve menu metadata.

What I learned.

Guardrails must be explicit, local, and testable. “Don’t hallucinate” is useless; “only recommend items that appear verbatim in data and cite them” works.

The highest leverage is output schemas plus UNKNOWN handling. They align UX, engineering, and safety with one move.

Asking fewer but better questions beats being conversational. Two targeted questions capture 80% of decision signal.